### Agent IV: RAG Agent - Practice & Exercises
In this notebook, you will practice building a Retrieval-Augmented Generation (RAG) agent that splits document text, loads it into a Chroma database, and routes queries to retrieval tools dynamically.

#### Part 1: Follow-Along Practice
##### GoalBuild a RAG agent querying details from the `Stock_Market_Performance_2024.pdf` document.

##### Steps:1. Load and chunk the PDF document (`Stock_Market_Performance_2024.pdf`).
2. Embed chunks using `OpenAIEmbeddings` and store them in a local Chroma vector database.
3. Define a retriever tool function (`retriever_tool`) and bind it to the LLM.
4. Set up a StateGraph containing an LLM call node, a retriever action node, and conditional routing edges.

In [ ]:
import os
from dotenv import load_dotenv
from typing import Annotated, Sequence, TypedDict
from langchain_core.messages import BaseMessage, SystemMessage, ToolMessage
from langgraph.graph.message import add_messages
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_core.tools import tool
from langgraph.graph import StateGraph, END

load_dotenv()

In [ ]:
# 1. Load and Chunk PDF
pdf_path = "Stock_Market_Performance_2024.pdf"
if not os.path.exists(pdf_path):
    # Fallback/Mock content for vector store if local PDF is missing
    from langchain_core.documents import Document
    pages_split = [
        Document(page_content="In 2024, the tech sector gained 25% driven by AI breakthroughs."),
        Document(page_content="Energy stocks declined 5% due to oil oversupply issues.")
    ]
else:
    loader = PyPDFLoader(pdf_path)
    pages = loader.load()
    splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
    pages_split = splitter.split_documents(pages)

In [ ]:
# 2. Vector DB creation
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = Chroma.from_documents(
    documents=pages_split,
    embedding=embeddings,
    collection_name="stock_market"
)
retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 3})

In [ ]:
# 3. Define retriever tool and bind
@tool
def retriever_tool(query: str) -> str:
    """This tool searches and returns the information from the Stock Market Performance 2024 document."""
    docs = retriever.invoke(query)
    results = [f"Doc {i+1}: {d.page_content}" for i, d in enumerate(docs)]
    return "\n\n".join(results)

tools = [retriever_tool]
tools_dict = {t.name: t for t in tools}
llm = ChatOpenAI(model="gpt-4o", temperature=0).bind_tools(tools)

In [ ]:
# 4. Define graph layout
class AgentState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], add_messages]

def call_llm(state: AgentState) -> dict:
    sys = SystemMessage(content="Answer questions using the retriever tool.")
    msg = llm.invoke([sys] + list(state["messages"]))
    return {"messages": [msg]}

def take_action(state: AgentState) -> dict:
    tool_calls = state["messages"][-1].tool_calls
    results = []
    for t in tool_calls:
        res_content = tools_dict[t["name"]].invoke(t["args"].get("query", ""))
        results.append(ToolMessage(tool_call_id=t["id"], name=t["name"], content=str(res_content)))
    return {"messages": results}

def should_continue(state: AgentState):
    last_msg = state["messages"][-1]
    return "retriever_agent" if (hasattr(last_msg, "tool_calls") and last_msg.tool_calls) else "end"

In [ ]:
# 5. Build, compile and run
graph = StateGraph(AgentState)
graph.add_node("llm", call_llm)
graph.add_node("retriever_agent", take_action)
graph.set_entry_point("llm")
graph.add_conditional_edges(
    "llm",
    should_continue,
    {
        "retriever_agent": "retriever_agent",
        "end": END
    }
)
graph.add_edge("retriever_agent", "llm")
app = graph.compile()

In [ ]:
response = app.invoke({"messages": [("user", "How did the tech sector perform in 2024?")]})
print(response["messages"][-1].content)

---

#### Part 2: Practice Exercises
##### Exercise 1: Multi-Document RAG with Document Type Router
###### GoalAdd a second Mock File retriever or a specific search tool for corporate emails (e.g. searching about meeting details), allowing the model to choose the correct resource dynamically.

###### Sample Input```python
{"messages": [("user", "Check the meeting email: why was the user unable to attend?")]}
```

###### Sample OutputOutputs citing the email document (e.g., `"The user was unable to attend due to a conflicting doctor's appointment."`).

###### Plan1. **Define Second Tool:** Create a retriever tool `@tool` named `email_retriever` returning content from `Unable_to_Attend_Meeting_Email.txt`:
   - `"Subject: Unable to Attend Meeting\n\nHi, I cannot make it to the meeting today because I have a doctor's appointment scheduled at that time."`
2. **Bind Both Tools:** Bind `[retriever_tool, email_retriever]` to the LLM.
3. **Configure take_action:** Ensure the tool runner maps and invokes both tools correctly.
4. **Compile & Run:** Invoke with the sample query and confirm it picks `email_retriever`.

In [ ]:
# TODO: Define your State, Tools, Nodes, Graph, Compile, and Invoke it here


<details>
<summary>💡 Click to see solution</summary>

```python
from typing import Annotated, Sequence, TypedDict
from langchain_core.messages import BaseMessage, SystemMessage, ToolMessage
from langgraph.graph.message import add_messages
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langgraph.graph import StateGraph, START, END

### 1. Stateclass AgentState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], add_messages]

### 2. Tools@tool
def retriever_tool(query: str) -> str:
    """This tool searches the Stock Market Performance 2024 database."""
    return "In 2024, the tech sector gained 25% driven by AI breakthroughs."

@tool
def email_retriever(query: str) -> str:
    """This tool searches corporate email records."""
    return "Subject: Unable to Attend Meeting\n\nHi, I cannot make it to the meeting today because I have a doctor's appointment scheduled at that time."

tools = [retriever_tool, email_retriever]
tools_dict = {t.name: t for t in tools}
llm = ChatOpenAI(model="gpt-4o", temperature=0).bind_tools(tools)

### 3. Nodesdef call_llm(state: AgentState) -> dict:
    sys = SystemMessage(content="Use retriever_tool for stock performance, and email_retriever for email records.")
    msg = llm.invoke([sys] + list(state["messages"]))
    return {"messages": [msg]}

def take_action(state: AgentState) -> dict:
    tool_calls = state["messages"][-1].tool_calls
    results = []
    for t in tool_calls:
        res_content = tools_dict[t["name"]].invoke(t["args"].get("query", ""))
        results.append(ToolMessage(tool_call_id=t["id"], name=t["name"], content=str(res_content)))
    return {"messages": results}

def should_continue(state: AgentState):
    last_msg = state["messages"][-1]
    return "retriever_agent" if (hasattr(last_msg, "tool_calls") and last_msg.tool_calls) else "end"

### 4. Buildgraph = StateGraph(AgentState)
graph.add_node("llm", call_llm)
graph.add_node("retriever_agent", take_action)
graph.set_entry_point("llm")
graph.add_conditional_edges(
    "llm",
    should_continue,
    {
        "retriever_agent": "retriever_agent",
        "end": END
    }
)
graph.add_edge("retriever_agent", "llm")
app = graph.compile()

### 5. Testres = app.invoke({"messages": [("user", "Check the meeting email: why was the user unable to attend?")]})
print(res["messages"][-1].content)
```
</details>